# Aula 06 — Modelagem de Tópicos: LDA e BERTopic · acompanhamento

**Origem:** `IAST-ProcessamentodeLinguagemNatural-Fase04_aula06-v1.pptx` · **Kernel:** `.venv`

## O contrato

- **Eu escrevi:** as células do professor na ordem, o corpus que falta, o mapa de dependências e os avisos do que não roda.
- **Você escreve:** o porquê, nos blocos **Sua vez**.

## ⚠️ Nenhum dos 5 slides roda com o ambiente atual

São dois conjuntos de dependências, com custos bem diferentes:

**LDA (slides 19, 38, 58)** — instalação leve:

```powershell
.venv\Scripts\python -m pip install gensim pyLDAvis
```

**BERTopic (slides 53, 54, 58)** — instalação pesada, **~3 GB**, porque `sentence-transformers` puxa o PyTorch, e o modelo multilíngue baixa mais algumas centenas de MB no primeiro uso:

```powershell
.venv\Scripts\python -m pip install bertopic sentence-transformers
```

Não instalei nada. As perguntas de conceito não dependem de execução.

## O aviso que vale mais que os outros

Modelagem de tópicos é a técnica desta disciplina **mais próxima do seu Tech Challenge** — e a mais enganosa num corpus pequeno. O slide 19 pede 10 tópicos com `no_below=10`; o slide 53 pede `min_topic_size=20`. Esses números foram escritos para milhares de documentos. Nas 64 reviews que eu forneço, LDA vai produzir tópicos sem sentido e o BERTopic vai jogar quase tudo na categoria `-1` (outlier).

Isso não é defeito do notebook: é a lição. Você tem **33.812 reviews reais** do Olist preparadas no repo do Tech Challenge, e esta é a aula em que usá-las deixa de ser bônus e passa a ser a única forma de ver a técnica funcionar. A pergunta final do notebook é sobre isso.

In [ ]:
# [meu] Corpus e tokens_limpos — nenhum slide define `reviews` nem `tokens_limpos`.
# `tokens_limpos` é descrito no slide 19 como "output da Aula 2": lista de listas de tokens.
reviews = [
    'Não tive problema, produto bom e entrega rápida',
    'Gostei muito, não demorou para chegar',
    'Não veio quebrado, produto bom, recomendo muito',
    'Recomendo muito, não tive problema com a entrega',
    'Não demorou nada, qualidade boa, gostei muito',
    'Produto bom, não tive problema com o atendimento',
    'Gostei muito do produto, não veio quebrado',
    'Não demorou, entrega rápida, recomendo muito',
    'Não tive problema nenhum, qualidade boa',
    'Recomendo muito o produto, não demorou nada',
    'Não veio quebrado e chegou rápido, gostei muito',
    'Produto bom, gostei muito, não tive problema',
    'Não demorou para chegar, atendimento bom',
    'Gostei muito da qualidade, não veio quebrado',
    'Não tive problema com o vendedor, recomendo muito',
    'Entrega rápida, não demorou, produto bom',
    'Não veio quebrado, qualidade boa, gostei muito',
    'Recomendo muito, não tive problema nenhum',
    'Não demorou e o produto é bom, gostei muito',
    'Qualidade boa, não veio quebrado, recomendo muito',
    'Não tive problema, chegou rápido, produto bom',
    'Gostei muito, não veio quebrado, entrega rápida',
    'Não demorou nada, recomendo muito esse vendedor',
    'Produto bom, não tive problema, qualidade boa',
    'Não veio quebrado, gostei muito do atendimento',
    'Recomendo muito, entrega rápida, não demorou',
    'Não tive problema com a qualidade, produto bom',
    'Gostei muito, chegou rápido, não demorou',
    'Não gostei, produto veio quebrado',
    'Não recomendo, entrega demorou muito',
    'Não gostei do produto, tive problema com a entrega',
    'Não recomendo, qualidade ruim, veio quebrado',
    'Não gostei, demorou muito para chegar',
    'Não recomendo esse produto, tive problema',
    'Não gostei da qualidade, veio quebrado',
    'Não recomendo, produto ruim, demorou muito',
    'Não gostei do atendimento, tive problema',
    'Não recomendo, veio quebrado e demorou',
    'Não gostei, tive problema com o vendedor',
    'Não recomendo a loja, qualidade ruim',
    'Não gostei do produto bom que veio quebrado',
    'Não recomendo, demorou muito e veio quebrado',
    'Não gostei, qualidade ruim, tive problema',
    'Não recomendo, produto veio quebrado',
    'Não gostei da entrega, demorou muito',
    'Não recomendo, tive problema com a qualidade',
    'Não gostei, veio quebrado, atendimento ruim',
    'Não recomendo, demorou muito para chegar',
    'Não gostei do vendedor, produto veio quebrado',
    'Não recomendo, tive problema com o produto',
    'Não gostei, demorou e veio quebrado',
    'Não recomendo, qualidade ruim e demorou muito',
    'Não gostei da qualidade, tive problema',
    'Não recomendo, veio quebrado, entrega ruim',
    'Não gostei, produto ruim, demorou muito',
    'Não recomendo, tive problema com a entrega',
]
import re
tokens_limpos = [[t for t in re.findall(r'\w+', r.lower()) if len(t) > 2]
                 for r in reviews]
print(len(reviews), 'reviews |', sum(len(t) for t in tokens_limpos), 'tokens')
print('exemplo:', tokens_limpos[0])

---
# Bloco 1 · LDA (slides 19, 38)

## Slide 19 — LDA em Gensim, do BoW ao coherence

> ### 🔴 Precisa de `gensim`
> ```powershell
> .venv\Scripts\python -m pip install gensim
> ```
>
> ⚠️ Define `lda`, `corpus_bow` e `dicionario`, usados pelos slides 38 e 58.
>
> ⚠️ `filter_extremes(no_below=10)` exige que um termo apareça em 10 documentos. Com 64 reviews curtas, o dicionário vai ficar minúsculo — e `num_topics=10` sobre isso é mais tópicos que palavras úteis. Rode e veja; é a pergunta 3.

Antes de rodar, escreva sua previsão: quantas palavras você espera que sobrem no dicionário, e o que 10 tópicos vão parecer sobre elas?

In [ ]:
from gensim import corpora
from gensim.models import LdaModel
from gensim.models.coherencemodel import CoherenceModel
# 'tokens_limpos' = lista de listas de tokens (output Aula 2)
# Ex.: [["produto", "bom"], ["atendimento", "ruim"], ...]
# 1) Criar dicionário e corpus em BoW
dicionario = corpora.Dictionary(tokens_limpos)
dicionario.filter_extremes(no_below=10, no_above=0.5)
corpus_bow = [dicionario.doc2bow(d) for d in tokens_limpos]
# 2) Treinar LDA com 10 tópicos
lda = LdaModel(corpus=corpus_bow, id2word=dicionario, num_topics=10,
               passes=20, alpha="auto", eta="auto", random_state=42)
# 3) Inspecionar tópicos descobertos
for idx, topic in lda.print_topics(num_words=8):
    print(f"Tópico {idx}: {topic}")
# 4) Coerência (qualidade dos tópicos)
coh = CoherenceModel(model=lda, texts=tokens_limpos,
                     dictionary=dicionario, coherence="c_v")
print(f"Coherence (c_v): {coh.get_coherence():.3f}")

### Sua vez

1. `filter_extremes(no_below=10, no_above=0.5)` é exatamente o `min_df` e o `max_df` da aula 03, com outro nome. Qual é qual, e por que a biblioteca mudou a nomenclatura?
2. Cada tópico sai como uma lista de palavras com pesos, sem nome. **Quem** decide que o tópico 0 é "entrega"? O que isso implica sobre usar LDA num relatório automático?
3. Olhe o tamanho do dicionário depois do `filter_extremes` e compare com `num_topics=10`. O que significa pedir 10 tópicos quando o vocabulário tem essa dimensão — e o que a métrica de coerência vai dizer?

_

_

## Slide 38 — pyLDAvis

> ### 🔴 Precisa de `pyLDAvis` e do `lda` do slide 19
> ```powershell
> .venv\Scripts\python -m pip install pyLDAvis
> ```
>
> ⚠️ Grava `topicos_pyldavis.html` na pasta da aula.

In [ ]:
# pip install pyLDAvis
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis
# Preparar visualização (lda, corpus_bow, dicionario do slide anterior)
vis = gensimvis.prepare(lda, corpus_bow, dicionario)
pyLDAvis.save_html(vis, "topicos_pyldavis.html")
# Abrir o HTML no browser para exploração interativa.

### Sua vez

1. O título do slide diz que esta é "a ferramenta que vendeu LDA a cientistas de dados". Uma visualização não muda a qualidade do modelo. O que, então, ela de fato resolveu?
2. O pyLDAvis mostra tópicos como círculos num plano, com tamanho e distância. Depois do t-SNE da aula 04, que ressalva você já sabe que precisa fazer sobre distâncias em gráficos 2D de dados de alta dimensão?
3. No seu Tech Challenge, o vídeo de 7 minutos precisa convencer sobre decisões técnicas. Uma visualização dessas ajuda ou atrapalha, considerando que a nota avalia rastreabilidade e fundamentação?

_

_

---
# Bloco 2 · BERTopic (slides 53, 54)

## Slide 53 — BERTopic em PT-BR

> ### 🔴 Instalação pesada: `bertopic` + `sentence-transformers` (~3 GB com o PyTorch)
> ```powershell
> .venv\Scripts\python -m pip install bertopic sentence-transformers
> ```
> O modelo `paraphrase-multilingual-MiniLM-L12-v2` baixa mais algumas centenas de MB na primeira execução.
>
> ⚠️ `min_topic_size=20` em 64 documentos: quase tudo vai para o tópico `-1` (outlier). Isso é a pergunta 3.
>
> ⚠️ Define `topic_model` e `embedder`, usados pelos slides 54 e 58.

**Este slide é a ponte mais direta entre a disciplina e o Tech Challenge.** O `SentenceTransformer` é a primeira vez no curso que você vê um vetor por **documento**, e não por palavra — exatamente o que o seu índice de RAG precisa.

In [ ]:
# pip install bertopic sentence-transformers
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
# Modelo de embeddings em português (multilingual)
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
# BERTopic com configuração padrão
topic_model = BERTopic(
    embedding_model=embedder,
    language="multilingual",
    min_topic_size=20,        # ignora tópicos com menos de 20 docs
    nr_topics="auto",
)
# 'reviews' é uma lista de strings em PT
topics, probs = topic_model.fit_transform(reviews)
# Inspecionar tópicos
print(topic_model.get_topic_info().head(10))
# Topic  Count                          Name
#    -1   1245   -1_geral_outros_diversos
#     0    832     0_entrega_prazo_atraso
#     1    721     1_qualidade_produto_bom
#     2    615     2_atendimento_suporte_problema

### Sua vez

1. O BERTopic recebe as **strings cruas** — sem `limpar_normalizar`, sem remoção de stop words, sem stemming. O LDA do slide 19 recebeu `tokens_limpos`. Por que a diferença, e o que isso confirma sobre o que você decidiu no Tech Challenge quanto a ter dois pipelines de normalização?
2. Existe um tópico `-1`. O LDA não tem equivalente: ele atribui **todo** documento a alguma mistura de tópicos. Qual das duas atitudes é mais honesta, e por quê?
3. `min_topic_size=20` sobre 64 documentos. Rode e veja quantos sobraram fora do `-1`. Qual é a regra que liga esse parâmetro ao tamanho do corpus?

_

_

## Slide 54 — Visualizações nativas

> ### 🔴 Usa o `topic_model` do slide 53.
>
> ⚠️ Grava quatro arquivos `.html` na pasta da aula.

In [ ]:
# Mapa de tópicos em 2D (similar ao pyLDAvis)
fig = topic_model.visualize_topics()
fig.write_html("mapa_topicos.html")
# Barras horizontais com palavras-chave por tópico
fig = topic_model.visualize_barchart(top_n_topics=10)
fig.write_html("barras_topicos.html")
# Hierarquia de tópicos (dendrograma)
fig = topic_model.visualize_hierarchy(top_n_topics=20)
fig.write_html("hierarquia.html")
# Heatmap de similaridade entre tópicos
fig = topic_model.visualize_heatmap(top_n_topics=15)
fig.write_html("heatmap.html")
# Essas visualizações são especialmente úteis em apresentações
# para times de produto, marketing ou C-level que precisam
# entender o que está sendo discutido nos dados.

### Sua vez

1. O slide pede `top_n_topics=10`, `20` e `15` em três chamadas. Se o slide 53 produziu menos tópicos que isso, o que acontece? Teste a sua hipótese antes de rodar.
2. Um dendrograma de tópicos sugere que tópicos têm hierarquia. Essa hierarquia foi **descoberta nos dados** ou **imposta pelo algoritmo de clusterização**? A diferença importa quando alguém pergunta "então existem 3 grandes temas?".

_

_

---
# Bloco 3 · A comparação (slide 58)

## Slide 58 — BERTopic vs LDA

> ### 🔴 Precisa de tudo: `gensim` + `bertopic` + `sentence-transformers`, e das variáveis dos slides 19 e 53.

O slide reporta, num corpus de 10 mil reviews:

```
Modelo       Tempo (s)    Coherence
LDA          45.2         0.428
BERTopic     312.8        0.567   ← mais coerente, mais lento
```

Antes de rodar, escreva sua previsão para o corpus de 64 reviews: a ordem vai se manter?

In [ ]:
import time
from gensim.models import LdaModel
from gensim.models.coherencemodel import CoherenceModel
# 1) LDA via Gensim
t0 = time.time()
lda = LdaModel(corpus=corpus_bow, id2word=dicionario,
               num_topics=10, passes=20, random_state=42)
tempo_lda = time.time() - t0
coh_lda = CoherenceModel(model=lda, texts=tokens_limpos,
                         dictionary=dicionario, coherence="c_v").get_coherence()
# 2) BERTopic
t0 = time.time()
topic_model = BERTopic(embedding_model=embedder,
                       language="multilingual", min_topic_size=20)
topics, _ = topic_model.fit_transform(reviews)
tempo_bertopic = time.time() - t0
# Coerência manual: top words de cada tópico
top_words_bert = [[w for w, _ in topic_model.get_topic(t)[:10]]
                  for t in topic_model.get_topics() if t != -1]
coh_bert = CoherenceModel(topics=top_words_bert, texts=tokens_limpos,
                          dictionary=dicionario, coherence="c_v").get_coherence()
print(f"{'Modelo':<12} {'Tempo (s)':<12} {'Coherence':<12}")
print(f"{'LDA':<12} {tempo_lda:<12.1f} {coh_lda:<12.3f}")
print(f"{'BERTopic':<12} {tempo_bertopic:<12.1f} {coh_bert:<12.3f}")
# Saída em corpus de 10k reviews:
# LDA          45.2          0.428
# BERTopic     312.8         0.567   ← mais coerente, mais lento

### Sua vez

1. A coerência do BERTopic é calculada com o `dicionario` do LDA — construído a partir de `tokens_limpos`, que passou por limpeza. Mas o BERTopic trabalhou com texto cru. Essa comparação é justa? Aponte onde ela pende para um lado.
2. O slide conclui "mais coerente, mais lento" com 7× de diferença de tempo. Para o seu Tech Challenge, em que momento esse custo é pago — uma vez, ou a cada pergunta do usuário? Isso muda a decisão?
3. `c_v` é uma métrica automática de coerência. Ela mede se as palavras de um tópico costumam aparecer juntas. Isso é a mesma coisa que o tópico ser **útil** para um gestor? Dê um exemplo onde as duas coisas divergem.

_

_

---
## Fechamento — responda sem olhar para cima

1. Em uma frase cada: o que o LDA assume sobre como um documento é gerado, e o que o BERTopic faz no lugar disso.
2. As aulas 03, 04 e 06 produziram três representações de um documento: contagem esparsa, média de vetores de palavras, e vetor de sentença. Ordene-as pela quantidade de informação que preservam do texto original e justifique a ordem.
3. Esta foi a última aula de NLP. Liste o que você sabe fazer hoje que não sabia na aula 01 — e, mais importante, o que continua sem resposta depois das seis aulas.

_

_

## Ponte para o Tech Challenge — a mais importante do curso

4. O `SentenceTransformer` do slide 53 é um modelo de embedding de **sentença**. É exatamente a peça que o seu índice precisa, e é a primeira vez que ela aparece. Escreva, com o que você sabe hoje, o que falta entre ter esse modelo e ter um sistema que responde perguntas com evidência.
5. Modelagem de tópicos responde "sobre o que as pessoas falam?". O seu desafio precisa responder "o que as pessoas dizem sobre X, e com que evidência?". São perguntas diferentes. Topic modeling tem algum papel no seu projeto, ou é um caminho paralelo? Decida e justifique — essa decisão entra no README.
6. O corpus de 64 reviews deste notebook não permite ver nenhuma dessas técnicas funcionando. Você tem 33.812 reviews reais preparadas. Vale rodar LDA e BERTopic sobre elas antes de começar o RAG — ou é desvio de rota?

_

_